In [ ]:
# remove the # from the second line if you haven't pandas dataframe already installed on your env
!pip install pyTMD
#!pip install pandas

In [11]:
import pyTMD 
import numpy as np
import pandas as pd
def tidal_elevations(amp_list,phase_list,start_date:str,end_date:str,H0:float=0):
    """
    This is a function to predict water leveld based on 4 principal astronomical constituents, namely: M2,S2,K1 and O1
    amp_list (list): a list of tidal constituents amplitudes in meters. The order of values MUST match the constituents_list.
    phase_list (list): a list of tidal constituents phases in degrees. The order of values MUST match the constituents_list.
    start_date (str): start date of the generated water levels.  format: yyyy-mm-dd hh:mm 
    end_date (str): end date of the generated water levels. format: yyyy-mm-dd hh:mm
    H0 (float): The mean water level in the location. (0 corresponds to Mean sea level)
    return:
    water levels (numpy array)
    """
    # Input your basic constituent amplitudes and phases
    constituents_list = ['m2', 's2', 'k1', 'o1']  # Plain lowercase strings
    amp_array=np.array(amp_list)
    phase_array=np.array(phase_list)
    times = pd.date_range(start=start_date, end=end_date, freq='10min')

    # Convert time to "Days relative to Jan 1, 1992" (Modified Julian Days baseline)
    epoch = np.datetime64('1992-01-01T00:00:00')
    t_days = (times.to_numpy() - epoch) / np.timedelta64(1, 'D')

    # FETCH THE 3 COEFFICIENTS
    pu, pf, G_astro = pyTMD.constituents.arguments(t_days, constituents_list)

    # EXECUTE THE HYDRODYNAMIC FORMULA
    # Formula components: 
    # pf      = Nodal scaling multiplier factor (f)
    # G_astro = Astronomical background argument (V0) in degrees
    # pu      = Nodal correction phase angle (u) in radians -> convert to degrees
    mean_sea_level = H0  # Baseline offset (H0)
    water_elevation = np.zeros(len(times)) + mean_sea_level
    ## Hint: put H0 0 at first, and see the RMSE between generated elevations and measured Elevations, then come back here and change H0 to RMSE value...
    ## To have better correspondence between generated and measured curves 
    # Loop over each constituent to add its wave component
    for i, name in enumerate(constituents_list):
        f_i = pf[:, i]                # Nodal amplitude scale factor (f)
        v_i = G_astro[:, i]           # Astronomical Argument (V0) in degrees
        u_i = np.degrees(pu[:, i])    # Nodal phase adjustment (u) converted from radians to degrees
    
        A_i = amp_array[i]
        G_i = phase_array[i]
    
    # Combined phase angle matching: V0 + u - G
        total_phase_deg = v_i + u_i - G_i
    
    # Synthesize component height contribution
        component_tide = f_i * A_i * np.cos(np.radians(total_phase_deg))
        water_elevation += component_tide
    return water_elevation

In [12]:
amplitudes_Arvand_IHO = [0.841, 0.286, 0.497, 0.298]    # meters (A)
phases_Arvand_IHO = [221.4, 278.6, 250.3, 205.3]  # degrees (G)

amplitudes_FAO_IHO = [0.824, 0.253, 0.438, 0.254]   # meters (A)
phases_FAO_IHO = [250, 309, 270, 226]  # degrees (G)

amplitudes_admirality = [0.84, 0.29, 0.5, 0.3]   # meters (A)
phases_admirality = [308, 9, 295, 247]   # degrees (G)

amplitudes_TPXO7_2 = [0.741, 0.24, 0.477, 0.286]    # meters (A)
phases_TPXO7_2 = [210, 268, 254, 210]  # degrees (G)

amplitudes_article = [0.799, 0.249, 0.449, 0.261]    # meters (A)
phases_article = [278.58, 36.59, 305.53, 218.89]   # degrees (G)

start='2017-08-07 06:50'
end='2017-09-12 06:20'

In [13]:
elevations_Arvand_IHO = tidal_elevations(amplitudes_Arvand_IHO,phases_Arvand_IHO,start,end,1.86) 
elevations_Arvand_IHO

array([3.21007653, 3.22579843, 3.233856  , ..., 1.03600872, 1.03519589,
       1.04047173], shape=(5182,))